# 02 Data pipeline

Solar and load are real, openly licensed measurements at 5-minute resolution:

- **Solar:** rooftop PV in Utrecht, the Netherlands (Visser et al., 2022, doi:10.5281/zenodo.10953360, CC BY 4.0).
- **Load:** ElectricityLoadDiagrams20112014 (Trindade, 2015, UCI, doi:10.24432/C58C86, CC BY 4.0).

The two sources come from different places and years, so **solar and load share no real weather**. Gaps are never filled; a window that touches missing data is skipped. The time split has no overlap.
Only the training and validation parts are shown here; the test part is sealed.

```{include} ../_caveats.md
```

In [ ]:
from IPython.display import Markdown, display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from common import *
m2 = load("m2_summary.json")
display(Markdown(provenance(m2, "m2_summary.json")))
display(Markdown("> " + m2["note"]))
t = m2["config"]["time"]
def span(split):
    start, end = pd.Timestamp(t[split][0]), pd.Timestamp(t[split][1]) - pd.Timedelta(days=1)
    off = pd.DateOffset(years=t["solar_year_offset"][split])
    return (f"load from {start:%Y-%m-%d} to {end:%Y-%m-%d} and solar from {start + off:%Y-%m-%d} to {end + off:%Y-%m-%d}")
display(Markdown("Solar and load are paired by month and day, while the years differ: training uses " + span("train") +
                 "; validation uses " + span("val") + "."))

## Split

In [ ]:
rows = []
for split, label in [("train", "training"), ("val", "validation")]:
    p = m2["per_split"][split]
    rows.append({"part": label, "scenario clock": f"{p['first_step']} → {p['last_step']}",
                 "solar data": f"{p['solar_first'][:10]} → {p['solar_last'][:10]}", "steps": f"{p['steps']:,}",
                 "valid share": pct(p["valid_share"], 1), "full windows": f"{p['full_windows']:,}",
                 "feeder load peak at ×1.0 (MW)": fmt(p["feeder_load_peak_mw_at_x1"], 3),
                 "mean (MW)": fmt(p["feeder_load_mean_mw_at_x1"], 3), "extreme-ramp steps": f"{p['extreme_steps']:,}"})
table(rows, index="part")

## Load and solar processing

In [ ]:
L, S = m2["load"], m2["solar"]
rows = [
    {"item": "load clients kept / total", "value": f"{L['clients_used']} / {L['clients_total']}"},
    {"item": "bus peak / case33bw peak on training (min-max)",
     "value": f"{fmt(L['train_bus_peak_over_case33bw_min'], 2)} - {fmt(L['train_bus_peak_over_case33bw_max'], 2)}"},
    {"item": "largest client share within a bus (median, max)",
     "value": f"{pct(L['largest_client_share_within_a_bus_median'], 0)}, {pct(L['largest_client_share_within_a_bus_max'], 0)}"},
    {"item": "PV systems kept / total", "value": f"{S['systems_usable']} / {S['systems_total']}"},
    {"item": "PV systems per PV bus", "value": ", ".join(map(str, S["systems_per_bus"]))},
    {"item": "training total PV max / mean (MW)", "value": f"{fmt(S['train_max_total_mw'], 3)} / {fmt(S['train_mean_total_mw'], 3)}"},
]
table(rows, index="item")

## Ramp rate and extreme events

r_t is the change of total feeder PV per minute. A step is "extreme" if |r_t| is at least the 99th percentile of all valid
training steps (night included), so about 2% of daytime steps are extreme.

In [ ]:
R = m2["ramp"]
rc = load("m2_ramp_check.json")
rows = [
    {"item": "threshold (MW per minute), training part", "value": fmt(R["threshold_mw_per_min_from_train"], 4)},
    {"item": "same threshold as % of PV capacity per 5 min", "value": f"{fmt(R['threshold_in_percent_of_pv_capacity_per_5_min'], 2)}%"},
    {"item": "largest |r_t| on training (MW per minute)", "value": fmt(R["train_max_abs_ramp_mw_per_min"], 3)},
    {"item": "extreme share of daytime steps (training)", "value": pct(rc["extreme_share_of_daytime_steps"])},
    {"item": "falling ramps among extreme steps", "value": pct(rc["falling_share_of_extreme_steps"], 1)},
    {"item": "extreme steps where the set of reporting PV systems changed", "value": pct(rc["reporting_set_changes_share_extreme_steps"], 1)},
    {"item": "of these, still extreme with the same set of systems", "value": pct(rc["extreme_steps_also_extreme_with_same_set"], 1)},
]
table(rows, index="item")

## Voltages without control (training part)

The PV size of 3.9 MW was chosen with this kind of no-control check on the training part only, before any controller
existed: at 2.6 MW no training step exceeded 1.05 pu.

In [ ]:
rows = []
for r in m2["train_no_control_check"]:
    rows.append({"load scale": r["load_scale"], "steps": r["steps_checked"], "V min (pu)": fmt(r["v_min"], 4),
                 "V max (pu)": fmt(r["v_max"], 4), "steps with a bus < 0.95": pct(r["share_of_steps_with_any_bus_below_0.95"]),
                 "steps with a bus > 1.05": pct(r["share_of_steps_with_any_bus_above_1.05"])})
table(rows, index="load scale")

## Do extreme ramps and violations coincide? (training part, no control)

This check is why RQ2 was reworded on 2026-11-04 to "improve voltage margin and reduce voltage violations".

In [ ]:
rq = load("m2_rq2_check.json")
rows = []
for reg, label in REGIMES.items():
    r = rq["regimes"][reg]
    a, c, e = r["requested"]["violation_among_extreme_steps"], r["for_comparison"], r["events"]
    rows.append({"regime": label, "extreme steps with a violation": f"{a['count']} of {a['of']} ({pct(a['share'], 1)})",
                 "daytime steps with a violation": pct(c["violation_among_daytime_steps"]["share"], 1),
                 "days with an extreme step": e["days_with_extreme_step"],
                 "of which with a violation at it": e["days_with_extreme_step_and_violation_at_it"]})
table(rows, index="regime")

## RQ2 metrics without a controller (training part)

The RQ2 metrics (defined on 2026-11-04, before any controller existed) are shown here only for the no-control case on the
training part, as context. They have not been computed for any controller.

In [ ]:
mb = load("m2_margin_baseline.json")
rows = []
for reg, label in REGIMES.items():
    p = mb["regimes"][reg]["primary_over_ramp_windows"]
    sec = mb["regimes"][reg]["secondary_violations_at_extreme_step"]
    ci = lambda d, n=4: f"{fmt(d['mean'], n)} [{fmt(d['mean_ci'][0], n)}, {fmt(d['mean_ci'][1], n)}]"
    rows.append({"regime": label, "windows used (excluded)": f"{p['windows_used']} ({p['windows_excluded_incomplete']})",
                 "worst margin, mean [95% CI] (pu)": ci(p["worst_margin_pu"]),
                 "max deviation, mean [95% CI] (pu)": ci(p["max_deviation_pu"]),
                 "minutes to safe band, mean [95% CI]": ci(p["minutes_to_safe_band"], 1),
                 "censored share": pct(p["censored_share_never_inside_in_window"], 1),
                 "violations at the extreme step": f"{sec['steps']} steps, {sec['distinct_days_with_violation']} days"})
table(rows, index="regime")